<a href="https://colab.research.google.com/github/jjkiljanski/biebrza-shrub-encroachment-analysis/blob/main/biebrza_streaming_1997_2015_inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Streaming Conv1D Encroachment Prediction from Google Earth Engine (1997–2015)

This notebook:

1. Initializes Google Earth Engine and Google Drive in Colab.
2. Builds a 10-step (1997–2015) biannual Landsat time-series image (6 bands per step).
3. Creates a **tile index** over the area of interest (AOI).
4. Streams each tile from GEE → runs your Conv1D model → saves prediction tiles to Google Drive.
5. Is **resumable**: if Colab disconnects, simply rerun the notebook and it will continue from the next unprocessed tile.

The model expects input of shape `(batch, T, C)` with:
- `T = 10` time steps (1997–2015 biannual)
- `C = 6` bands (`NDMI, NBR, NIR, NDVI, SWIR1, SWIR2`)
- `num_classes = 5`

Predictions are saved as **one-band GeoTIFF tiles** with the predicted class index (0–4) in Google Drive.
You can later mosaic these prediction tiles into a full-park encroachment risk map.

In [8]:
# Install required packages (Colab)
!pip install -q earthengine-api geemap rasterio torch torchvision tqdm

In [9]:
import os
import json
import math

import ee
import geemap
import torch
import torch.nn as nn
import rasterio
from rasterio.transform import from_origin
import numpy as np
from tqdm import tqdm
from google.colab import drive

# Mount Google Drive (for prediction tiles + tile index)
drive.mount('/content/drive')

# Authenticate and initialize Earth Engine
ee.Authenticate()
ee.Initialize(project='biebrza-encroachment-analysis')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Define Area of Interest (AOI) and Build 1997–2015 Time-Series Image

⚠️ **IMPORTANT:** Replace the AOI asset ID below with your own geometry (e.g., full Biebrza NP or Lower Biebrza).
It must be an `ee.FeatureCollection` or `ee.Geometry` defining the area where you want predictions.

In [10]:
import ipywidgets as widgets
from datetime import datetime

# ============================================================
# Load Biebrzański National Park boundary
#    (from the WDPA – World Database on Protected Areas)
# ============================================================

wdpa = ee.FeatureCollection("WCMC/WDPA/current/polygons")

# Filter areas whose NAME contains "Biebrza" (safe way to match spelling)
bpn = wdpa.filter(ee.Filter.stringContains("NAME", "Biebrza"))

print("Number of matching park polygons:", bpn.size().getInfo())

# Geometry for clipping satellite images
aoi = bpn.geometry()

print('AOI bounds (lon/lat):', aoi.bounds().coordinates().getInfo())

Number of matching park polygons: 3
AOI bounds (lon/lat): [[[22.39636562061044, 53.19907326752765], [23.597246250349325, 53.19907326752765], [23.597246250349325, 53.80556002887963], [22.39636562061044, 53.80556002887963], [22.39636562061044, 53.19907326752765]]]


In [11]:
# Build 1997–2015 biannual image stack (10 steps x 6 bands = 60 bands)
project_root = 'projects/biebrza-encroachment-analysis/assets/image_composites'
bands = ['NDMI', 'NBR', 'NIR', 'NDVI', 'SWIR1', 'SWIR2']

# Biannual years for Window A: 1997, 1999, ..., 2015
windowA_years = list(range(1997, 2016, 2))  # 10 time steps

def load_biannual_image(year):
    asset_id = f"{project_root}/biannual_{year}_{year+1}"
    img = ee.Image(asset_id).select(bands)
    # Rename bands with year prefix, e.g. '1997_NDVI'
    renamed = img.rename([f"{year}_{b}" for b in bands])
    return renamed

images = [load_biannual_image(y) for y in windowA_years]
windowA_img = ee.Image.cat(images).clip(aoi)

# Inspect projection & scale
proj = windowA_img.projection()
crs = proj.crs().getInfo()
scale = proj.nominalScale().getInfo()

print('Window A years:', windowA_years)
print('CRS:', crs)
print('Scale (m):', scale)
print('Number of bands:', windowA_img.bandNames().size().getInfo())
print('First 10 bands:', windowA_img.bandNames().slice(0, 10).getInfo())

Window A years: [1997, 1999, 2001, 2003, 2005, 2007, 2009, 2011, 2013, 2015]
CRS: EPSG:4326
Scale (m): 10
Number of bands: 60
First 10 bands: ['1997_NDMI', '1997_NBR', '1997_NIR', '1997_NDVI', '1997_SWIR1', '1997_SWIR2', '1999_NDMI', '1999_NBR', '1999_NIR', '1999_NDVI']


## Create / Load Tile Index (Resumable)

We tile the AOI into ~256×256-pixel patches at the native scale, and store a JSON file in Drive
tracking which tiles are **done**. If Colab disconnects, you just rerun the notebook and it will
continue from the next unprocessed tile.

Tile JSON schema:
```json
{
  "meta": {"tile_pixels": 256, "scale": 30, ...},
  "tiles": [
    {"id": 0, "lon_min": ..., "lat_min": ..., "lon_max": ..., "lat_max": ..., "done": false},
    ...
  ]
}
```

In [12]:
# Where to store the tile index and prediction tiles in Drive
pred_base_dir = '/content/drive/MyDrive/biebrza_preds'
os.makedirs(pred_base_dir, exist_ok=True)

tile_json_path = os.path.join(pred_base_dir, 'windowA_tiles.json')
tile_pred_dir = os.path.join(pred_base_dir, 'windowA_tiles')
os.makedirs(tile_pred_dir, exist_ok=True)

TILE_PIXELS = 256  # ~256x256 pixel tiles

def create_tile_index(image, aoi, tile_pixels, json_path):
    """Create a tile index over the AOI and save it to JSON in Drive.
    Tiles are defined in lon/lat (EPSG:4326) as rectangles covering the AOI.
    """
    # AOI bounds in lon/lat
    bounds = aoi.bounds().coordinates().getInfo()[0]
    lons = [pt[0] for pt in bounds]
    lats = [pt[1] for pt in bounds]
    min_lon, max_lon = min(lons), max(lons)
    min_lat, max_lat = min(lats), max(lats)

    # Center latitude for lon-degree calculation
    center_lat = 0.5 * (min_lat + max_lat)
    center_lat_rad = math.radians(center_lat)

    # Use image scale to approximate tile size in meters
    proj = image.projection()
    pixel_scale = proj.nominalScale().getInfo()  # meters per pixel
    tile_size_m = tile_pixels * pixel_scale

    # Convert meters to degrees (approximate)
    meters_per_deg_lat = 111320.0
    meters_per_deg_lon = meters_per_deg_lat * math.cos(center_lat_rad)

    lat_step = tile_size_m / meters_per_deg_lat
    lon_step = tile_size_m / meters_per_deg_lon if meters_per_deg_lon != 0 else tile_size_m / meters_per_deg_lat

    tiles = []
    tile_id = 0
    lat = min_lat
    while lat < max_lat:
        next_lat = min(lat + lat_step, max_lat)
        lon = min_lon
        while lon < max_lon:
            next_lon = min(lon + lon_step, max_lon)
            tiles.append({
                'id': tile_id,
                'lon_min': lon,
                'lat_min': lat,
                'lon_max': next_lon,
                'lat_max': next_lat,
                'done': False,
            })
            tile_id += 1
            lon = next_lon
        lat = next_lat

    state = {
        'meta': {
            'tile_pixels': tile_pixels,
            'pixel_scale_m': pixel_scale,
            'crs': proj.crs().getInfo(),
            'aoi_bounds': {
                'min_lon': min_lon,
                'max_lon': max_lon,
                'min_lat': min_lat,
                'max_lat': max_lat,
            },
        },
        'tiles': tiles,
    }

    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(state, f, indent=2)

    print(f'Created tile index with {len(tiles)} tiles and saved to {json_path}')
    return state

if os.path.exists(tile_json_path):
    print('Loading existing tile index from:', tile_json_path)
    with open(tile_json_path, 'r', encoding='utf-8') as f:
        tiles_state = json.load(f)
else:
    print('No existing tile index found. Creating a new one...')
    tiles_state = create_tile_index(windowA_img, aoi, TILE_PIXELS, tile_json_path)

tiles = tiles_state['tiles']
num_tiles = len(tiles)
num_done = sum(1 for t in tiles if t.get('done'))
print(f'Total tiles: {num_tiles}, already done: {num_done}')

Loading existing tile index from: /content/drive/MyDrive/biebrza_preds/windowA_tiles.json
Total tiles: 864, already done: 0


## Load Conv1D Model

This cell loads the Conv1D classifier you trained. Make sure `conv1d_best_model.pth`
is uploaded to `/content/` in this Colab session (or change `model_path`).

In [13]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

class Conv1DClassifier(nn.Module):
    def __init__(self, seq_len, num_classes, in_channels):
        super().__init__()
        self.seq_len = seq_len
        self.conv1 = nn.Conv1d(in_channels=in_channels, out_channels=32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(in_channels=32,        out_channels=64, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.pool = nn.AdaptiveAvgPool1d(1)  # global average pooling over time
        self.fc   = nn.Linear(64, num_classes)

    def forward(self, x):
        # x: [B, T, C]
        x = x.permute(0, 2, 1)   # [B, C, T]
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        x = self.pool(x).squeeze(-1)  # [B, 64]
        logits = self.fc(x)
        return logits

T = 10
C = 6
num_classes = 5

# Path to your trained weights (upload this file to /content first)
model_path = '/content/conv1d_best_model.pth'

model = Conv1DClassifier(seq_len=T, num_classes=num_classes, in_channels=C)
model.load_state_dict(torch.load(model_path, map_location=device))
model.to(device)
model.eval()

print('Model loaded successfully!')

Using device: cpu
Model loaded successfully!


## Helper Functions: Download Tile from GEE, Run Model, Save Prediction

Each tile is processed as follows:
1. Use `geemap.ee_export_image` to download a small multi-band GeoTIFF tile from GEE to `/content`.
2. Read it with `rasterio` into a `(B, H, W)` NumPy array (with `B = 60` bands).
3. Reshape to `(H·W, T, C)` and run the Conv1D model in batches.
4. Take `argmax` over classes to get the predicted class index per pixel (0–4).
5. Save the prediction tile as a one-band GeoTIFF to Google Drive.

The tile's `done` flag is then set to `True` and the JSON index is updated
so that the process can be **resumed** later.

In [14]:
def download_tile_from_ee(tile, image, scale, tmp_dir='/content'):
    """Download a small tile from GEE as a GeoTIFF and return (data, profile).
    data shape: (bands, H, W)
    """
    lon_min, lat_min = tile['lon_min'], tile['lat_min']
    lon_max, lat_max = tile['lon_max'], tile['lat_max']
    tile_id = tile['id']

    geom = ee.Geometry.Rectangle([lon_min, lat_min, lon_max, lat_max])
    tmp_path = os.path.join(tmp_dir, f'windowA_tile_{tile_id:05d}.tif')

    # Export a small tile (should be well under EE's request-size limit)
    geemap.ee_export_image(
        image,
        filename=tmp_path,
        scale=scale,
        region=geom,
        file_per_band=False,
    )

    if not os.path.exists(tmp_path):
        raise RuntimeError(f'Failed to download tile {tile_id} to {tmp_path}')

    with rasterio.open(tmp_path) as src:
        data = src.read()  # (bands, H, W)
        profile = src.profile

    # Clean up input tile to save space
    try:
        os.remove(tmp_path)
    except OSError:
        pass

    return data, profile


def run_model_on_tile_array(tile_data, model, T=10, C=6, num_classes=5, inner_batch_size=4096):
    """
    Run the Conv1D model on a tile array.

    tile_data: numpy array (B, H, W) with B = T * C = 60

    Returns three numpy arrays of shape (H, W):
      - dominant_class: argmax over classes (float32, values 0..4)
      - p_encroachment: probability of class 'wetland_to_woody' (index 4)
      - uncertainty: 1 - max_class_probability
    """
    B, H, W = tile_data.shape
    expected_B = T * C
    if B != expected_B:
        raise ValueError(f'Expected {expected_B} bands, got {B}')

    # Reshape to (T, C, H, W) assuming step-major band order
    tile_tc_hw = tile_data.reshape(T, C, H, W)
    # Move to (H, W, T, C)
    tile_hw_tc = np.transpose(tile_tc_hw, (2, 3, 0, 1))  # (H, W, T, C)
    # Flatten spatial dims -> (N, T, C)
    N = H * W
    tile_n_tc = tile_hw_tc.reshape(N, T, C)

    x = torch.from_numpy(tile_n_tc).float().to(device)

    all_pred_classes = []
    all_max_probs = []
    all_enc_probs = []

    encroachment_class_idx = 4  # wetland_to_woody

    model.eval()
    with torch.no_grad():
        for start in range(0, N, inner_batch_size):
            end = min(start + inner_batch_size, N)
            batch = x[start:end]  # (batch_size, T, C)
            logits = model(batch)  # (batch_size, num_classes)
            probs = torch.softmax(logits, dim=1)  # (batch_size, num_classes)

            max_probs, pred_classes = probs.max(dim=1)            # (batch_size,)
            enc_probs = probs[:, encroachment_class_idx]          # (batch_size,)

            all_pred_classes.append(pred_classes.cpu())
            all_max_probs.append(max_probs.cpu())
            all_enc_probs.append(enc_probs.cpu())

    # Concatenate all batches
    all_pred_classes = torch.cat(all_pred_classes, dim=0).numpy()   # (N,)
    all_max_probs    = torch.cat(all_max_probs, dim=0).numpy()      # (N,)
    all_enc_probs    = torch.cat(all_enc_probs, dim=0).numpy()      # (N,)

    # Reshape back to (H, W)
    dominant_class = all_pred_classes.reshape(H, W).astype("float32")
    p_encroachment = all_enc_probs.reshape(H, W).astype("float32")
    max_prob_hw    = all_max_probs.reshape(H, W).astype("float32")
    uncertainty    = 1.0 - max_prob_hw

    return dominant_class, p_encroachment, uncertainty


def process_single_tile(tile, image, model, scale, pred_dir,
                        T=10, C=6, num_classes=5, inner_batch_size=4096):
    """
    Download a tile from GEE, run model, save prediction tile to Drive.

    Writes a 3-band GeoTIFF:
      band 1: dominant class (0..4, stored as float32)
      band 2: p(wetland_to_woody)
      band 3: uncertainty = 1 - max_class_prob
    """
    tile_id = tile['id']
    print(f'Processing tile {tile_id}...')

    tile_data, profile = download_tile_from_ee(tile, image, scale)
    dom_class, p_enc, uncertainty = run_model_on_tile_array(
        tile_data,
        model,
        T=T,
        C=C,
        num_classes=num_classes,
        inner_batch_size=inner_batch_size,
    )

    # Create output raster profile: 3 bands, float32
    out_profile = profile.copy()
    out_profile.update(
        driver='GTiff',
        count=3,
        dtype='float32',
        compress='lzw',
    )

    pred_path = os.path.join(pred_dir, f'pred_windowA_tile_{tile_id:05d}.tif')
    with rasterio.open(pred_path, 'w', **out_profile) as dst:
        # band 1: dominant class (0..4)
        dst.write(dom_class.astype('float32'), 1)
        # band 2: p(wetland_to_woody)
        dst.write(p_enc.astype('float32'), 2)
        # band 3: uncertainty = 1 - max_prob
        dst.write(uncertainty.astype('float32'), 3)

    print(f'  Saved prediction tile to {pred_path}')
    return pred_path


## Main Processing Loop (Resumable)

This will:
1. Iterate over all tiles in the JSON index.
2. Skip tiles where `done == True`.
3. For each remaining tile: download → predict → save prediction to Drive.
4. Mark `tile['done'] = True` and write the updated JSON back to Drive after each tile.

If Colab disconnects, just rerun all cells above **and then rerun this cell**.
The loop will continue from the next unfinished tile.

In [ ]:
num_tiles = len(tiles)
num_done = sum(1 for t in tiles if t.get('done'))
print(f'Starting processing loop. Total tiles: {num_tiles}, already done: {num_done}')

for tile in tiles:
    if tile.get('done'):
        continue

    try:
        _ = process_single_tile(
            tile=tile,
            image=windowA_img,
            model=model,
            scale=scale,
            pred_dir=tile_pred_dir,
            T=T,
            C=C,
            num_classes=num_classes,
            inner_batch_size=4096,
        )
        tile['done'] = True
    except Exception as e:
        print(f'Error processing tile {tile["id"]}: {e}')
        # Optionally, you can break here or just continue to next tile
        # break
    finally:
        # Update the JSON file after each tile to preserve progress
        with open(tile_json_path, 'w', encoding='utf-8') as f:
            json.dump(tiles_state, f, indent=2)

num_done = sum(1 for t in tiles if t.get('done'))
print(f'Processing finished (or loop ended). Tiles done: {num_done} / {num_tiles}')

Starting processing loop. Total tiles: 864, already done: 12
Processing tile 12...
Generating URL ...
Please wait ...


Data downloaded to /content/windowA_tile_00012.tif
  Saved prediction tile to /content/drive/MyDrive/biebrza_preds/windowA_tiles/pred_windowA_tile_00012.tif
Processing tile 13...
Generating URL ...
Please wait ...


Data downloaded to /content/windowA_tile_00013.tif
  Saved prediction tile to /content/drive/MyDrive/biebrza_preds/windowA_tiles/pred_windowA_tile_00013.tif
Processing tile 14...
Generating URL ...
Please wait ...


Data downloaded to /content/windowA_tile_00014.tif
  Saved prediction tile to /content/drive/MyDrive/biebrza_preds/windowA_tiles/pred_windowA_tile_00014.tif
Processing tile 15...
Generating URL ...
Please wait ...


Data downloaded to /content/windowA_tile_00015.tif
  Saved prediction tile to /content/drive/MyDrive/biebrza_preds/windowA_tiles/pred_windowA_tile_00015.tif
Processing tile 16...
Generating URL ...
Please wait ...


Data downloaded to /content/windowA_tile_00016.tif
  Saved prediction tile to /content/drive/MyDrive/biebrza_preds/windowA_tiles/pred_windowA_tile_00016.tif
Processing tile 17...
Generating URL ...
Please wait ...


Data downloaded to /content/windowA_tile_00017.tif
  Saved prediction tile to /content/drive/MyDrive/biebrza_preds/windowA_tiles/pred_windowA_tile_00017.tif
Processing tile 18...
Generating URL ...
Please wait ...


Data downloaded to /content/windowA_tile_00018.tif
  Saved prediction tile to /content/drive/MyDrive/biebrza_preds/windowA_tiles/pred_windowA_tile_00018.tif
Processing tile 19...
Generating URL ...


## (Optional) Mosaic Prediction Tiles into a Single Raster

Once all tiles are processed (`done == True` for all), you can merge them into a single
encroachment map. This step can be memory-intensive depending on your AOI size, so
you may want to run it on a smaller AOI first (e.g., Lower Biebrza).

In [ ]:
from glob import glob
from rasterio.merge import merge

# Read all prediction tiles
pred_tile_paths = sorted(glob(os.path.join(tile_pred_dir, 'pred_windowA_tile_*.tif')))
print(f'Found {len(pred_tile_paths)} prediction tiles for mosaicking.')

if len(pred_tile_paths) == 0:
    print('No prediction tiles found. Make sure you ran the processing loop and tiles are saved.')
else:
    src_files_to_mosaic = [rasterio.open(p) for p in pred_tile_paths]
    mosaic_array, mosaic_transform = merge(src_files_to_mosaic)  # (bands, H, W)
    # Keep profile from first tile
    mosaic_profile = src_files_to_mosaic[0].profile.copy()
    for src in src_files_to_mosaic:
        src.close()

    print("Mosaic array shape:", mosaic_array.shape)  # should be (3, H, W)

    mosaic_profile.update(
        transform=mosaic_transform,
        height=mosaic_array.shape[1],
        width=mosaic_array.shape[2],
        count=3,
        dtype='float32',
        compress='lzw',
    )

    mosaic_path = os.path.join(pred_base_dir, 'pred_windowA_mosaic_3band.tif')
    with rasterio.open(mosaic_path, 'w', **mosaic_profile) as dst:
        dst.write(mosaic_array)

    print('Mosaic saved to:', mosaic_path)
    print('Bands:')
    print('  1: dominant class (0..4)')
    print('  2: p(wetland_to_woody)')
    print('  3: uncertainty = 1 - max_prob')